In [5]:
import os

for root, dirs, files in os.walk("C:/"):
    if "chromedriver.exe" in files:
        print(os.path.join(root, "chromedriver.exe"))


C:/Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.119\chromedriver.exe
C:/Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.68\chromedriver.exe
C:/Users\Jaydeb\.cache\selenium\chromedriver\win64\137.0.7151.70\chromedriver.exe
C:/Users\Jaydeb\.cache\selenium\chromedriver\win64\138.0.7204.49\chromedriver.exe
C:/Users\Jaydeb\.cache\selenium\chromedriver\win64\138.0.7204.92\chromedriver.exe
C:/Users\Jaydeb\.wdm\drivers\chromedriver\win64\137.0.7151.119\chromedriver-win32\chromedriver.exe
C:/Users\Jaydeb\.wdm\drivers\chromedriver\win64\137.0.7151.68\chromedriver-win32\chromedriver.exe
C:/Users\Jaydeb\.wdm\drivers\chromedriver\win64\137.0.7151.70\chromedriver-win32\chromedriver.exe
C:/Users\Jaydeb\.wdm\drivers\chromedriver\win64\138.0.7204.49\chromedriver-win32\chromedriver.exe
C:/Users\Jaydeb\.wdm\drivers\chromedriver\win64\138.0.7204.92\chromedriver-win32\chromedriver.exe
C:/Users\Jaydeb\anaconda3\Lib\site-packages\chromedriver_autoinstaller\137\chromedriver.exe


In [1]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"

def extract_tender_details(soup):
    # Sector (Tender Type)
    sector = ""
    sector_tag = soup.find("h2", class_="mb-0")
    if sector_tag:
        sector = sector_tag.get_text(strip=True)

    # Location
    location = ""
    location_tag = soup.find("span", string=lambda x: x and ',' in x)
    if location_tag:
        location = location_tag.get_text(strip=True)

    # Work Description (find the first <span> after location with long text)
    work_desc = ""
    if location_tag:
        sibling = location_tag.find_next_sibling("span")
        while sibling:
            txt = sibling.get_text(strip=True)
            if len(txt) > 20 and "Opening Date" not in txt:
                work_desc = txt
                break
            sibling = sibling.find_next_sibling("span")

    # Opening Date, Closing Date, Tender Amount
    opening_date = ""
    closing_date = ""
    tender_amount = ""

    all_spans = soup.find_all("span", class_="d-block list-heading")
    for span in all_spans:
        label = span.get_text(strip=True)
        if label == "Opening Date":
            next_span = span.find_next_sibling("span")
            if next_span:
                opening_date = next_span.get_text(strip=True)
        elif label == "Closing Date":
            next_span = span.find_next_sibling("span")
            if next_span:
                closing_date = next_span.get_text(strip=True)
        elif label == "Tender Amount":
            badge = span.find_next_sibling("span", class_="badge")
            if badge:
                tender_amount = badge.get_text(strip=True)
            else:
                next_span = span.find_next_sibling("span")
                if next_span:
                    tender_amount = next_span.get_text(strip=True)
    # Fallback for amount: find badge if not set
    if not tender_amount:
        badge = soup.find("span", class_="badge")
        if badge:
            tender_amount = badge.get_text(strip=True)

    return sector, location, work_desc, opening_date, closing_date, tender_amount

def scrape_tendershark(query, max_pages):
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    service = Service(CHROMEDRIVER_PATH)
    driver = webdriver.Chrome(service=service, options=chrome_options)

    base_url = f"https://www.tendershark.com/all-tenders/active?query={query}&page={{}}"
    results = []

    for page in range(1, max_pages + 1):
        print(f"[{query}] Scraping listing page {page}/{max_pages}")
        driver.get(base_url.format(page))
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        cards = soup.find_all("div", class_="border-0 d-flex flex-column pb-0 col-sm-12 col-12 p-0")
        for card in cards:
            link_tag = card.find("a", href=True)
            project_url = link_tag["href"] if link_tag else ""
            if not project_url.startswith("http"):
                project_url = "https://www.tendershark.com" + project_url

            # Now scrape the detail page
            driver.get(project_url)
            time.sleep(2)
            detail_soup = BeautifulSoup(driver.page_source, "html.parser")

            sector, location, work_desc, opening_date, closing_date, tender_amount = extract_tender_details(detail_soup)

            # Optional: Also extract summary/description if needed (like in your earlier script)
            summary = ""
            for tag in detail_soup.find_all("p", title=True):
                summary = tag.get("title", "")
                if summary:
                    break

            results.append({
                "Keyword": query,
                "Tender URL": project_url,
                "Sector": sector,
                "Location": location,
                "Work Description": work_desc,
                "Summary": summary,
                "Opening Date": opening_date,
                "Closing Date": closing_date,
                "Tender Amount": tender_amount,
            })
    driver.quit()
    return results

# --- SCRAPE ALL KEYWORDS TO ONE SHEET ---
all_results = []
all_results += scrape_tendershark("JUIDCO", 2)

# (You can add more queries here, e.g. all_results += scrape_tendershark("ductile iron", 17))

df = pd.DataFrame(all_results)
df.to_excel("tendershark_ALL_cleaned.xlsx", index=False)
print("DONE! All data saved to tendershark_ALL_JUIDCO.xlsx")


[JUIDCO] Scraping listing page 1/2
[JUIDCO] Scraping listing page 2/2
DONE! All data saved to tendershark_ALL_JUIDCO.xlsx


In [ ]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"

def get_max_pages(soup):
    page_info = soup.find("span", class_="page-info")
    if page_info:
        match = re.search(r'Page \d+ of (\d+)', page_info.text)
        if match:
            return int(match.group(1))
    return 1

def extract_tender_details(soup):
    # Department (Sector)
    sector = ""
    sector_tag = soup.find("h2", class_="mb-0")
    if sector_tag:
        sector = sector_tag.get_text(strip=True)

    # Location
    location = ""
    location_tag = soup.find("span", string=lambda x: x and ',' in x)
    if location_tag:
        location = location_tag.get_text(strip=True)

    # Work Description
    work_desc = ""
    for p in soup.find_all("p", title=True):
        if "Work Description" in p.text:
            desc_span = p.find("span")
            if desc_span:
                work_desc = desc_span.get_text(strip=True)
                break

    # Tender Summary / Description (the next long span with "tender" in it)
    summary = ""
    all_spans = soup.find_all("span")
    for s in all_spans:
        txt = s.get_text(strip=True)
        if len(txt) > 40 and "tender" in txt.lower() and not any(x in txt for x in ["Opening Date", "Closing Date", "Tender Amount"]):
            summary = txt
            break

    # Dates and Amount
    opening_date, closing_date, tender_amount = "", "", ""
    for li in soup.find_all("li", class_="list-group-item"):
        spans = li.find_all("span", class_="d-block list-heading")
        if spans and len(spans) >= 2:
            label = spans[0].get_text(strip=True)
            value = spans[1].get_text(strip=True)
            if label == "Opening Date":
                opening_date = value
            elif label == "Closing Date":
                closing_date = value
            elif label == "Tender Amount":
                badge = li.find("span", class_="badge")
                tender_amount = badge.get_text(strip=True) if badge else value

    return sector, location, work_desc, summary, opening_date, closing_date, tender_amount

def scrape_tendershark(query):
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    service = Service(CHROMEDRIVER_PATH)
    driver = webdriver.Chrome(service=service, options=chrome_options)

    base_url = f"https://www.tendershark.com/all-tenders/active?query={query}&page={{}}"
    results = []

    # Get first page and detect max pages
    driver.get(base_url.format(1))
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    max_pages = get_max_pages(soup)
    print(f"Total pages found: {max_pages}")

    for page in range(1, max_pages + 1):
        print(f"[{query}] Scraping listing page {page}/{max_pages}")
        if page > 1:
            driver.get(base_url.format(page))
            time.sleep(2)
            soup = BeautifulSoup(driver.page_source, "html.parser")
        cards = soup.find_all("div", class_="border-0 d-flex flex-column pb-0 col-sm-12 col-12 p-0")
        for card in cards:
            link_tag = card.find("a", href=True)
            project_url = link_tag["href"] if link_tag else ""
            if not project_url.startswith("http"):
                project_url = "https://www.tendershark.com" + project_url

            # Now scrape the detail page
            driver.get(project_url)
            time.sleep(2)
            detail_soup = BeautifulSoup(driver.page_source, "html.parser")

            # Extract details
            sector, location, work_desc, summary, opening_date, closing_date, tender_amount = extract_tender_details(detail_soup)

            results.append({
                "Keyword": query,
                "Tender URL": project_url,
                "Sector": sector,
                "Location": location,
                "Work Description": work_desc,
                "Summary": summary,
                "Opening Date": opening_date,
                "Closing Date": closing_date,
                "Tender Amount": tender_amount,
            })
    driver.quit()
    return results

# --------- MAIN RUN ---------

all_results = []
all_results += scrape_tendershark("JUIDCO")  # <- Change/add keywords here as needed

df = pd.DataFrame(all_results)
df.to_excel("tendershark_ALL_cleaned1.xlsx", index=False)
print("DONE! All data saved to tendershark_ALL_cleaned1.xlsx")


Total pages found: 1274
[JUIDCO] Scraping listing page 1/1274


In [1]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"

def extract_tender_details(soup):
    sector = ""
    sector_tag = soup.find("h2", class_="mb-0")
    if sector_tag:
        sector = sector_tag.get_text(strip=True)

    location = ""
    location_tag = soup.find("span", string=lambda x: x and ',' in x)
    if location_tag:
        location = location_tag.get_text(strip=True)

    work_desc = ""
    for p in soup.find_all("p", title=True):
        if "Work Description" in p.text:
            desc_span = p.find("span")
            if desc_span:
                work_desc = desc_span.get_text(strip=True)
                break

    summary = ""
    all_spans = soup.find_all("span")
    for s in all_spans:
        txt = s.get_text(strip=True)
        if len(txt) > 40 and "tender" in txt.lower() and not any(x in txt for x in ["Opening Date", "Closing Date", "Tender Amount"]):
            summary = txt
            break

    opening_date, closing_date, tender_amount = "", "", ""
    for li in soup.find_all("li", class_="list-group-item"):
        spans = li.find_all("span", class_="d-block list-heading")
        if spans and len(spans) >= 2:
            label = spans[0].get_text(strip=True)
            value = spans[1].get_text(strip=True)
            if label == "Opening Date":
                opening_date = value
            elif label == "Closing Date":
                closing_date = value
            elif label == "Tender Amount":
                badge = li.find("span", class_="badge")
                tender_amount = badge.get_text(strip=True) if badge else value

    return sector, location, work_desc, summary, opening_date, closing_date, tender_amount

def scrape_tendershark(query):
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    service = Service(CHROMEDRIVER_PATH)
    driver = webdriver.Chrome(service=service, options=chrome_options)

    base_url = "https://www.tendershark.com/all-tenders/active?search=&query={}&page={}"
    results = []
    page = 1

    while True:
        print(f"[{query}] Scraping listing page {page}")
        driver.get(base_url.format(query, page))
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        cards = soup.find_all("div", class_="border-0 d-flex flex-column pb-0 col-sm-12 col-12 p-0")
        if not cards:
            print(f"No cards found on page {page}. Stopping.")
            break

        for card in cards:
            link_tag = card.find("a", href=True)
            project_url = link_tag["href"] if link_tag else ""
            if not project_url.startswith("http"):
                project_url = "https://www.tendershark.com" + project_url

            driver.get(project_url)
            time.sleep(2)
            detail_soup = BeautifulSoup(driver.page_source, "html.parser")

            sector, location, work_desc, summary, opening_date, closing_date, tender_amount = extract_tender_details(detail_soup)

            results.append({
                "Keyword": query,
                "Tender URL": project_url,
                "Sector": sector,
                "Location": location,
                "Work Description": work_desc,
                "Summary": summary,
                "Opening Date": opening_date,
                "Closing Date": closing_date,
                "Tender Amount": tender_amount,
            })

        # Pagination: check for "Next" button and if it's not disabled
        pagination_div = soup.find("div", class_="pagination")
        has_next = False
        if pagination_div:
            next_btn = pagination_div.find("a", class_="right-pagination-arrow")
            if next_btn and "disabled" not in next_btn.get("class", []):
                has_next = True
        if not has_next:
            print("No more 'Next' page. Stopping.")
            break
        page += 1

    driver.quit()
    return results

# --------- MAIN RUN ---------

all_results = []
all_results += scrape_tendershark("JUIDCO")  # You can add more keywords here

df = pd.DataFrame(all_results)
df.to_excel("tendershark_ALL_JUIDCO.xlsx", index=False)
print("DONE! All data saved to tendershark_ALL_JUIDCO.xlsx")


[JUIDCO] Scraping listing page 1
No more 'Next' page. Stopping.
DONE! All data saved to tendershark_ALL_JUIDCO.xlsx


In [2]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

CHROMEDRIVER_PATH = r"C:/Users/Jaydeb/.cache/selenium/chromedriver/win64/138.0.7204.92/chromedriver.exe"

def extract_listing_card(card):
    link_tag = card.find("a", href=True)
    project_url = link_tag["href"] if link_tag else ""
    if not project_url.startswith("http"):
        project_url = "https://www.tendershark.com" + project_url

    sector = ""
    sector_tag = card.find("h2", class_="mb-0")
    if sector_tag:
        sector = sector_tag.get_text(strip=True)

    location = ""
    loc_div = card.find("div", class_="location")
    if loc_div:
        location_span = loc_div.find("span")
        if location_span:
            location = location_span.get_text(strip=True)

    work_desc = ""
    for p in card.find_all("p", title=True):
        if "Work Description" in p.text:
            desc_span = p.find("span")
            if desc_span:
                work_desc = desc_span.get_text(strip=True)
                break

    summary = ""
    ps = card.find_all("p", title=True)
    for p in ps:
        if "Work Description" not in p.text:
            summary = p.get_text(strip=True)
            break

    return sector, location, work_desc, summary, project_url

def scrape_tendershark_listing(query):
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    service = Service(CHROMEDRIVER_PATH)
    driver = webdriver.Chrome(service=service, options=chrome_options)

    base_url = "https://www.tendershark.com/all-tenders/active?search=&query={}&page={}"
    results = []
    page = 1

    while True:
        print(f"[{query}] Scraping listing page {page}")
        driver.get(base_url.format(query, page))
        time.sleep(2)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        cards = soup.find_all("div", class_="border-0 d-flex flex-column pb-0 col-sm-12 col-12 p-0")
        if not cards:
            print(f"No cards found on page {page}. Stopping.")
            break

        for card in cards:
            sector, location, work_desc, summary, project_url = extract_listing_card(card)
            results.append({
                "Keyword": query,
                "Tender URL": project_url,
                "Sector": sector,
                "Location": location,
                "Work Description": work_desc,
                "Summary": summary,
            })

        pagination_div = soup.find("div", class_="pagination")
        has_next = False
        if pagination_div:
            next_btn = pagination_div.find("a", class_="right-pagination-arrow")
            if next_btn and "disabled" not in next_btn.get("class", []):
                has_next = True
        if not has_next:
            print("No more 'Next' page. Stopping.")
            break
        page += 1

    driver.quit()
    return results

# --------- MAIN RUN ---------

all_results = []
all_results += scrape_tendershark_listing("JUIDCO")

df = pd.DataFrame(all_results)
df.to_excel("tendershark_ALL_cleaned_listing.xlsx", index=False)
print("DONE! All data saved to tendershark_ALL_cleaned_listing.xlsx")


[JUIDCO] Scraping listing page 1
No more 'Next' page. Stopping.
DONE! All data saved to tendershark_ALL_cleaned_listing.xlsx
